# الدرس 03: بناء خط أنابيب RAG متكامل باستخدام LCEL
### Building Complete RAG Pipeline with LangChain Expression Language (LCEL)

في المحاضرة السابقة استخدمنا دوال جاهزة مثل `create_retrieval_chain` و `create_stuff_documents_chain`.
في هذه المحاضرة (**المحاضرة 28**)، سنبني نفس المسار من الصفر خطوة بخطوة باستخدام **لغة تعبير لانج تشين (LCEL)** ومشغل الربط (`|` Pipe Operator).

**المفاهيم الأساسية التي سنتعلمها:**
1. **`StrOutputParser`**: تحويل مخرجات النموذج مباشرة إلى نص (`str`).
2. **`RunnablePassthrough`**: تمرير المدخلات (مثل السؤال) مباشرة عبر السلسلة دون تعديل.
3. **`format_docs`**: دالة مخصصة لدمج المستندات المسترجعة في نص واحد (بديل شفاف لـ Stuff Chain).
4. **مشغل الربط (`|`)**: ربط المسترجع، القالب، النموذج، والمحلل في مسار تنفيذي متكامل وسريع التدفق (Streaming).

In [ ]:
# 1 import libraries
import os
import shutil
from dotenv import load_dotenv

load_dotenv()

from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableParallel

print("All libraries imported successfully!")

---
## 1. تجهيز البيانات والمستندات (Data Ingestion)

In [ ]:
# sample documents
docs = {
    "document_0.txt": (
        "Machine learning is a subset of artificial intelligence that learns from data without explicit programming. "
        "Supervised learning uses labeled datasets with algorithms like regression, decision trees, and support vector machines. "
        "Unsupervised learning finds hidden patterns in unlabeled data using clustering techniques."
    ),
    "document_1.txt": (
        "Deep learning is a specialized branch of machine learning inspired by artificial neural networks with multiple layers. "
        "Convolutional Neural Networks (CNNs) are best used for computer vision and image processing tasks. "
        "Recurrent Neural Networks (RNNs) and LSTMs excel at sequential and time-series data. Training relies on backpropagation."
    ),
    "document_2.txt": (
        "Natural language processing (NLP) enables machines to understand, interpret, and generate human language. "
        "The Transformer architecture revolutionized NLP in 2017 with self-attention mechanisms, replacing older recurrence. "
        "Pre-trained models like BERT and GPT set state-of-the-art benchmarks in translation, summarization, and question answering."
    )
}

sample_dir = "./chroma_sample_data"
os.makedirs(sample_dir, exist_ok=True)

for filename, content in docs.items():
    with open(os.path.join(sample_dir, filename), "w", encoding="utf-8") as f:
        f.write(content)

# load and split documents
loader = DirectoryLoader(sample_dir, glob="*.txt", loader_cls=TextLoader, loader_kwargs={"encoding": "utf-8"})
raw_documents = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
documents = splitter.split_documents(raw_documents)
print(f"Loaded {len(raw_documents)} raw docs -> Split into {len(documents)} chunks")

---
## 2. إعداد قاعدة المتجهات والمسترجع (ChromaDB Vector Store & Retriever)

In [ ]:
# initialize embedding model
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

persist_directory = "./chroma_vector_store"
collection_name = "rag_mastery_collection"

# reset collection if exists to avoid duplicate chunks
if os.path.exists(persist_directory):
    try:
        temp_db = Chroma(persist_directory=persist_directory, embedding_function=embeddings, collection_name=collection_name)
        temp_db.delete_collection()
    except Exception:
        pass
    shutil.rmtree(persist_directory, ignore_errors=True)

vector_store = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    persist_directory=persist_directory,
    collection_name=collection_name
)

# create retriever
retriever = vector_store.as_retriever(search_kwargs={"k": 3})
print(f"Vector store initialized. Stored documents: {vector_store._collection.count()}")
print(f"Retriever ready with k=3")

---
## 3. دالة دمج المستندات (`format_docs`)
هذه الدالة البسيطة هي بالضبط ما كانت تفعله دالة `create_stuff_documents_chain`:
تأخذ قائمة المستندات المسترجعة وتدمج محتواها النصي بفاصل سطرين (`\n\n`).

In [ ]:
# helper function to combine retrieved chunks into a single context string
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# test format_docs on retriever output
sample_docs = retriever.invoke("What is deep learning?")
formatted_sample = format_docs(sample_docs)
print("Sample Formatted Context (First 150 chars):")
print(formatted_sample[:150] + "...")

---
## 4. تصميم قالب التوجيه (Custom Prompt Template)

In [ ]:
# custom prompt template
template = """You are a helpful AI assistant. Use the following context to answer the question. 
If you don't know the answer based on the context, say you don't know.
Keep the answer concise and accurate.

Context:
{context}

Question:
{question}

Answer:"""

prompt = ChatPromptTemplate.from_template(template)
print(f"Prompt input variables: {prompt.input_variables}")

---
## 5. تهيئة نموذج اللغة الكبير (LLM)

In [ ]:
# initialize Groq LLM
llm = ChatGroq(
    api_key=os.getenv("GROQ_API_KEY"),
    model="openai/gpt-oss-20b",
    temperature=0.2,
    max_tokens=512
)

print(f"LLM initialized: {llm.model_name}")

---
## 6. بناء مسار الـ RAG بلغة LCEL (`rag_chain_lcel`)

لاحظ كيف يتم بناء السلسلة باستخدام مشغل الـ Pipe (`|`):
```python
{
    "context": retriever | format_docs,
    "question": RunnablePassthrough()
}
| prompt
| llm
| StrOutputParser()
```
- **`retriever | format_docs`**: يأخذ السؤال، يبحث في ChromaDB، ثم يمرر الـ Documents لدالة `format_docs` لتوضع في `{context}`.
- **`RunnablePassthrough()`**: يمرر نفس السؤال النصي كما هو دون تعديل ليوضع في `{question}`.
- **`prompt`**: يدمج الـ context والـ question داخل القالب.
- **`llm`**: يولد الإجابة.
- **`StrOutputParser()`**: يستخرج النص الصافي من رسالة `AIMessage`.

In [ ]:
# build the rag chain using lcel
rag_chain_lcel = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("LCEL RAG chain created successfully!")

---
## 7. اختبار وتشغيل السلسلة (`invoke`)
بفضل `RunnablePassthrough`، يمكننا تمرير السؤال كنص مباشر (`str`) بدلاً من تمرير قاموس معقد!

In [ ]:
# test rag pipeline with simple string query
question = "What is deep learning and how does it differ from traditional machine learning?"
response = rag_chain_lcel.invoke(question)

print(f"Question: {question}\n")
print(f"Answer:\n{response}")

---
## 8. ميزة البث المباشر الفوري (Streaming with LCEL)
من أقوى ميزات LCEL مقارنة بالسلاسل القديمة هي الدعم المدمج للبث الحرفي للكلمات فور توليدها (`stream`).

In [ ]:
# stream response token by token
stream_question = "What are CNNs best used for?"
print(f"Streaming answer for: '{stream_question}'\n")

for chunk in rag_chain_lcel.stream(stream_question):
    print(chunk, end="", flush=True)
print()

---
## 9. دالة متكاملة لعرض الإجابة والمصادر المسترجعة (`query_rag_lcel`)
كما شرح المحاضر، يمكننا عرض كل من الإجابة التوليدية والمستندات المسترجعة جنباً إلى جنب.

In [ ]:
# helper function to query rag and inspect retrieved chunks
def query_rag_lcel(query: str):
    print(f"\n{'='*65}")
    print(f"❓ Question: {query}")
    
    # 1 generate answer via lcel chain
    answer = rag_chain_lcel.invoke(query)
    print(f"\n💬 Answer:\n{answer}")
    
    # 2 retrieve source documents
    retrieved_docs = retriever.invoke(query)
    print(f"\n📚 Retrieved Chunks ({len(retrieved_docs)}): ")
    for i, doc in enumerate(retrieved_docs, start=1):
        src = os.path.basename(doc.metadata.get("source", "unknown"))
        print(f"   [{i}] ({src}): {doc.page_content[:80]}...")

# test multiple questions
query_rag_lcel("What is NLP and what revolutionized it in 2017?")
query_rag_lcel("What are the main types of machine learning?")
query_rag_lcel("Who won the World Cup in 2022?")

## 🎯 ملخص الدرس 03 (المحاضرة 28):
1. استبدلنا الدوال الجاهزة المغلقة (`create_retrieval_chain` و `create_stuff_documents_chain`) ببنية **LCEL** واضحة وشفافة.
2. استخدمنا دالة `format_docs` لدمج النصوص المسترجعة في `{context}`.
3. استخدمنا `RunnablePassthrough()` لتمرير السؤال المباشر كنص دون تغليفه في قواميس.
4. استخدمنا `StrOutputParser()` لتحويل مخرجات النموذج لنصوص نقية.
5. استعرضنا ميزة الـ `stream` المباشرة والمدعومة تلقائياً في LCEL.

**المحاضرة التالية (29):** كيفية **تحديث فهرس المتجهات وإضافة مستندات جديدة (`Updating Vector Store`)** إلى قاعدة بيانات موجودة مسبقاً دون إعادة بنائها من الصفر!